# Symbolic grounding: Whisper transcripts to PDDL predicates

This notebook evaluates whether `google/gemma-4-E4B-it` can ground a Whisper
transcript into a single PDDL predicate of the form `(at user LOCATION_ID)`.
Two prompt variants are compared head to head on two test sets.

## Test sets

| Set | Size | What it contains |
|-|-|-|
| Canonical | 20 | Clean, well-formed transcripts that match the expected phrasing. |
| Robustness | 31 | Realistic variation: spelled-out numbers, phonetic building codes, synonyms, filler words, reordered phrasings. |

## Prompt variants

| Variant | What it is |
|-|-|
| `basic` | Minimal single-line prompt that only states the output format. The naive baseline. |
| `optimized` | Structured prompt with explicit output and naming rules, the closed set of allowed location IDs, and 6 few-shot examples whose IDs are disjoint from both test sets, so gains reflect rule following rather than memorisation. |

## How to run (Colab)

1. Runtime -> A100 (E4B in bfloat16 needs about 16 GB VRAM).
2. Upload `manifest.csv`, `manifest_robust.csv` and `prompts.json` to `/content/`.
3. Run section 1 only; the runtime restarts at its end so the pinned Pillow loads cleanly.
4. Paste your `HF_TOKEN` in section 2 and run sections 2 to 9 in order.
5. The last cell zips `/content/results/` and downloads it.

## 1. Install dependencies

Pillow is pinned to avoid a known Colab conflict with the `_Ink` module. The
runtime restarts at the end of the cell so the new version loads cleanly; wait
for it to reconnect, then continue from section 2.

In [ ]:
!pip install -q "Pillow==10.4.0"
!pip install -q -U \
    "transformers>=4.47.0" \
    accelerate \
    sentencepiece \
    pandas \
    numpy \
    nltk \
    bert-score \
    huggingface_hub

import os
os.kill(os.getpid(), 9)

## 2. Credentials

Gemma 4 E4B-it is a gated model, so the HF token needs access granted for
`google/gemma-4-E4B-it`.

In [ ]:
import os
from huggingface_hub import login, whoami

# === EDIT THIS ===
HF_TOKEN = ''  # paste your HuggingFace read token here
# ==================

if not HF_TOKEN:
    raise ValueError('Set HF_TOKEN at the top of this cell before running.')

os.environ['HF_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
print('Logged in as:', whoami()['name'])

## 3. Load Gemma 4 E4B-it

Loads the model once in bfloat16 (float32 on CPU) with a fixed seed. Decoding is
greedy (`do_sample=False`, `num_beams=1`), so the run is deterministic.

In [ ]:
import time
import random
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_ID = 'google/gemma-4-E4B-it'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Loading {MODEL_ID} on {DEVICE} (dtype=auto, seed={SEED})...')
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID, token=os.environ['HF_TOKEN'])
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=os.environ['HF_TOKEN'],
    dtype='auto' if DEVICE == 'cuda' else torch.float32,
    device_map='auto' if DEVICE == 'cuda' else None,
).eval()
MODEL_LOAD_TIME_S = round(time.time() - t0, 2)

n_params = sum(p.numel() for p in model.parameters())
N_PARAMS_BILLIONS = round(n_params / 1e9, 2)
MODEL_DTYPE = str(model.dtype)
print(f'Loaded in {MODEL_LOAD_TIME_S} s - {N_PARAMS_BILLIONS}B params (dtype={MODEL_DTYPE})')
if DEVICE == 'cuda':
    torch.cuda.reset_peak_memory_stats()
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')

## 4. Load datasets and prompts

Finds `manifest.csv`, `manifest_robust.csv` and `prompts.json` under `/content/`.
The two CSVs are merged into one DataFrame with a `dataset` column so metrics can
be reported per set.

In [ ]:
import json
from pathlib import Path
import pandas as pd

BASE_DIR = Path('/content')
RESULTS_DIR = BASE_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def find_file(name: str) -> Path:
    hits = [p for p in BASE_DIR.rglob(name)
            if 'results' not in p.parts and 'sample_data' not in p.parts]
    if not hits:
        raise FileNotFoundError(f'{name!r} not found anywhere under {BASE_DIR}')
    return hits[0]


canonical_path = find_file('manifest.csv')
robust_path    = find_file('manifest_robust.csv')
prompts_path   = find_file('prompts.json')
print(f'Loaded canonical dataset from:  {canonical_path}')
print(f'Loaded robustness dataset from: {robust_path}')
print(f'Loaded prompts from:            {prompts_path}')

canonical_df = pd.read_csv(canonical_path)
robust_df    = pd.read_csv(robust_path)
canonical_df['dataset'] = 'canonical'
robust_df['dataset']    = 'robustness'
dataset_df = pd.concat([canonical_df, robust_df], ignore_index=True)

with prompts_path.open('r', encoding='utf-8') as f:
    PROMPTS_DATA = json.load(f)

PROMPT_VARIANTS = ['basic', 'optimized']
DATASET_NAMES = ['canonical', 'robustness']
ALLOWED_LOCATION_IDS = set(PROMPTS_DATA['allowed_location_ids'])

print(f'\nTotal transcripts: {len(dataset_df)} ({len(canonical_df)} canonical + {len(robust_df)} robustness)')
print(dataset_df.groupby(['dataset', 'type']).size().to_string())
print(f'\nPrompt variants: {PROMPT_VARIANTS}')
print(f'Allowed location IDs: {len(ALLOWED_LOCATION_IDS)}')

## 5. Run inference

One forward pass per (transcript, prompt variant) pair: 51 transcripts x 2
variants = 102 inferences. The system prompt is held fixed across both variants,
so the only thing that changes is the user prompt template. `MAX_NEW_TOKENS = 32`
is enough for any atom and short enough to avoid trailing text after the closing
parenthesis. Latency is measured around `torch.cuda.synchronize()` so it reflects
GPU time, not Python overhead.

In [ ]:
SYSTEM_PROMPT = (
    'You are a strict format extractor for indoor navigation. '
    'Read the question, follow its rules exactly, and output only what the '
    'question asks. Never invent identifiers. Never add explanations or extra '
    'words. Stop generating immediately after the requested output is complete.'
)
MAX_NEW_TOKENS = 32


def render_prompt(variant: str, transcript: str) -> str:
    template = PROMPTS_DATA['prompts'][variant]['template']
    return template.replace('<TRANSCRIPT>', transcript)


def run_one(prompt_text: str) -> tuple[str, float]:
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
        {'role': 'user',   'content': [{'type': 'text', 'text': prompt_text}]},
    ]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
        enable_thinking=False,
    ).to(model.device)
    inputs = inputs.to(dtype=model.dtype)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
        )
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    latency_ms = (time.time() - t0) * 1000

    input_len = inputs['input_ids'].shape[-1]
    response = processor.decode(output[0][input_len:], skip_special_tokens=True).strip()
    return response, latency_ms


raw_records = []
total_t0 = time.time()
n_total = len(dataset_df) * len(PROMPT_VARIANTS)
i = 0
for _, row in dataset_df.iterrows():
    for variant in PROMPT_VARIANTS:
        i += 1
        rendered = render_prompt(variant, row['whisper_transcript'])
        try:
            response, latency_ms = run_one(rendered)
            ok, err = True, ''
        except Exception as e:
            response, latency_ms, ok, err = '', 0.0, False, repr(e)
        raw_records.append({
            'id': row['id'],
            'dataset': row['dataset'],
            'type': row['type'],
            'whisper_transcript': row['whisper_transcript'],
            'expected_location_id': row['expected_location_id'],
            'expected_pddl_atom': row['expected_pddl_atom'],
            'prompt_variant': variant,
            'response': response,
            'latency_ms': round(latency_ms, 1),
            'inference_ok': ok,
            'error': err,
        })
        elapsed = time.time() - total_t0
        eta = (elapsed / i) * (n_total - i)
        print(f'[{i:2d}/{n_total}] {row["dataset"]:10s} id={row["id"]} variant={variant:9s} {latency_ms:7.1f} ms | elapsed {elapsed:6.1f}s ETA {eta:6.1f}s')

raw_df = pd.DataFrame(raw_records)
raw_df.to_csv(RESULTS_DIR / 'per_transcript_responses.csv', index=False)

VRAM_PEAK_GB = round(torch.cuda.max_memory_allocated() / 1024**3, 2) if torch.cuda.is_available() else 0.0
TOTAL_INFERENCE_TIME_S = round(time.time() - total_t0, 2)
print(f'\nDone. {len(raw_df)} inferences in {TOTAL_INFERENCE_TIME_S}s. Peak VRAM: {VRAM_PEAK_GB} GB.')
print(f'Saved per-transcript responses to {RESULTS_DIR / "per_transcript_responses.csv"}')

## 6. Parse outputs and compute the grounding metrics

Per (transcript, prompt variant) pair:

| Metric | What it measures |
|-|-|
| `format_validity` | The output contains a `(at user X)` pattern recoverable by regex. |
| `format_validity_strict` | The output is exactly `(at user X)`, with no surrounding text. |
| `exact_pddl_match` | The recovered atom matches the expected atom exactly. |
| `location_id_accuracy` | The extracted `location_id` equals the ground-truth one. |
| `valid_location_rate` | The extracted `location_id` belongs to the allowed set in `prompts.json`. |

A malformed output counts as incorrect for every match-based metric. Lenient
parsing only decides whether the output was recoverable, not whether it was right.

In [ ]:
import re

PDDL_ATOM_RE        = re.compile(r"\(\s*at\s+user\s+([a-zA-Z_][a-zA-Z0-9_]*)\s*\)")
PDDL_ATOM_STRICT_RE = re.compile(r"^\(at user ([a-z_][a-z0-9_]*)\)$")


def parse_pddl(response):
    if not response:
        return False, None
    m = PDDL_ATOM_RE.search(response.strip())
    if not m:
        return False, None
    return True, m.group(1).lower()


def parse_pddl_strict(response):
    if not response:
        return False
    return bool(PDDL_ATOM_STRICT_RE.match(response.strip()))


scored_records = []
for r in raw_records:
    response = r["response"]
    expected_atom = r["expected_pddl_atom"]
    expected_id = r["expected_location_id"]

    format_valid, extracted_id = parse_pddl(response)
    format_valid_strict = parse_pddl_strict(response)

    extracted_atom = f"(at user {extracted_id})" if extracted_id else ""
    exact_pddl_match    = bool(format_valid and extracted_atom == expected_atom)
    location_id_correct = bool(format_valid and extracted_id == expected_id)
    valid_location_id   = bool(format_valid and extracted_id in ALLOWED_LOCATION_IDS)

    scored_records.append({
        **r,
        "extracted_location_id": extracted_id or "",
        "extracted_atom": extracted_atom,
        "format_valid": format_valid,
        "format_valid_strict": format_valid_strict,
        "exact_pddl_match": exact_pddl_match,
        "location_id_correct": location_id_correct,
        "valid_location_id": valid_location_id,
    })

scored_df = pd.DataFrame(scored_records)
scored_df.to_csv(RESULTS_DIR / "per_transcript_scored.csv", index=False)
print(f"Saved per-transcript scored results to {RESULTS_DIR / 'per_transcript_scored.csv'}")

# Summary by prompt variant, pooled across canonical and robustness.
summary_rows = []
for variant in PROMPT_VARIANTS:
    sub = scored_df[scored_df["prompt_variant"] == variant]
    latencies = sub.loc[sub["inference_ok"], "latency_ms"].values
    summary_rows.append({
        "prompt_variant": variant,
        "n": len(sub),
        "format_validity":         round(sub["format_valid"].mean(), 4),
        "format_validity_strict":  round(sub["format_valid_strict"].mean(), 4),
        "exact_pddl_match":        round(sub["exact_pddl_match"].mean(), 4),
        "location_id_accuracy":    round(sub["location_id_correct"].mean(), 4),
        "valid_location_rate":     round(sub["valid_location_id"].mean(), 4),
        "mean_ms":                 round(float(np.mean(latencies)), 1) if len(latencies) else 0.0,
        "p95_ms":                  round(float(np.percentile(latencies, 95)), 1) if len(latencies) else 0.0,
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RESULTS_DIR / "per_variant_summary.csv", index=False)

# Per-(variant, type) breakdown, also pooled across datasets.
type_rows = []
for variant in PROMPT_VARIANTS:
    for t in ["initial_location", "destination"]:
        sub = scored_df[(scored_df["prompt_variant"] == variant) & (scored_df["type"] == t)]
        if len(sub) == 0:
            continue
        type_rows.append({
            "prompt_variant": variant,
            "type": t,
            "n": len(sub),
            "exact_pddl_match":     round(sub["exact_pddl_match"].mean(), 4),
            "location_id_accuracy": round(sub["location_id_correct"].mean(), 4),
        })
type_df = pd.DataFrame(type_rows)
type_df.to_csv(RESULTS_DIR / "per_variant_per_type_summary.csv", index=False)

print(f"\nPer-variant summary (pooled over {len(dataset_df)} transcripts):")
print(summary_df.to_string(index=False))
print("\nPer-(variant, type) breakdown (pooled):")
print(type_df.to_string(index=False))

## 7. Text similarity metrics: BLEU, GLEU, METEOR, BERTScore

Text similarity metrics, reported next to the grounding metrics for comparability
with the STT benchmarks. They are easy to game on short PDDL atoms:
`(at user classroom_g_0_3)` and `(at user classroom_g_0_1)` differ by one
character but point to different rooms, yet both score as almost identical. So the
grounding correctness still rests on `exact_pddl_match` and `location_id_accuracy`
from section 6.

Two calibration notes:

- BLEU-3, not BLEU-4: a PDDL atom tokenizes to 3 whitespace tokens, so 4-grams do
  not exist and BLEU-4 collapses even on perfect matches. N=3 matches the fixed
  token count of the reference.
- BERTScore uses `rescale_with_baseline=True`, the calibration recommended by its
  authors, so identical texts score near 1.0 and unrelated texts near 0.0.

In [ ]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.gleu_score import sentence_gleu
from nltk.translate.meteor_score import meteor_score

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from bert_score import score as bertscore_score


def tokenize(s):
    return s.split()


smoother = SmoothingFunction().method1

bleu3_scores, gleu_scores, meteor_scores = [], [], []
references_for_bert, candidates_for_bert = [], []

for r in scored_records:
    ref = r["expected_pddl_atom"]
    cand = r["response"] if r["response"] else ""
    ref_tokens, cand_tokens = tokenize(ref), tokenize(cand)
    bleu = sentence_bleu([ref_tokens], cand_tokens, weights=(1/3, 1/3, 1/3, 0), smoothing_function=smoother) if cand_tokens else 0.0
    gleu = sentence_gleu([ref_tokens], cand_tokens) if cand_tokens else 0.0
    met  = meteor_score([ref_tokens], cand_tokens) if cand_tokens else 0.0
    bleu3_scores.append(bleu)
    gleu_scores.append(gleu)
    meteor_scores.append(met)
    references_for_bert.append(ref)
    candidates_for_bert.append(cand)

print("Computing BERTScore (downloads a transformer on first run)...")
bert_P, bert_R, bert_F1 = bertscore_score(
    candidates_for_bert, references_for_bert,
    lang="en", rescale_with_baseline=True, verbose=False,
)

scored_df["bleu3"]        = bleu3_scores
scored_df["gleu"]         = gleu_scores
scored_df["meteor"]       = meteor_scores
scored_df["bertscore_p"]  = bert_P.tolist()
scored_df["bertscore_r"]  = bert_R.tolist()
scored_df["bertscore_f1"] = bert_F1.tolist()
scored_df.to_csv(RESULTS_DIR / "per_transcript_scored.csv", index=False)

support_rows = []
for variant in PROMPT_VARIANTS:
    sub = scored_df[scored_df["prompt_variant"] == variant]
    support_rows.append({
        "prompt_variant": variant,
        "bleu3":        round(sub["bleu3"].mean(), 4),
        "gleu":         round(sub["gleu"].mean(), 4),
        "meteor":       round(sub["meteor"].mean(), 4),
        "bertscore_f1": round(sub["bertscore_f1"].mean(), 4),
    })
support_df = pd.DataFrame(support_rows)
support_df.to_csv(RESULTS_DIR / "per_variant_support_metrics.csv", index=False)

print(f"\nText similarity metrics (pooled over {len(dataset_df)} transcripts, mean):")
print(support_df.to_string(index=False))

## 8. Reproducibility log

A single JSON capturing the model, dataset sizes, GPU, software versions, seed,
decoding parameters and metrics, so the run can be reproduced exactly.

In [ ]:
import platform
import datetime
import transformers
import PIL

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capability = '.'.join(str(x) for x in torch.cuda.get_device_capability(0))
    gpu_total_vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
    cuda_version = torch.version.cuda
    cudnn_version = torch.backends.cudnn.version()
else:
    gpu_name = 'CPU'
    gpu_capability = None
    gpu_total_vram_gb = 0.0
    cuda_version = None
    cudnn_version = None

reproducibility = {
    'run_timestamp_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'experiment': 'text_to_pddl_validation',
    'model': {
        'id': MODEL_ID,
        'n_params_billions': N_PARAMS_BILLIONS,
        'dtype': MODEL_DTYPE,
        'load_time_s': MODEL_LOAD_TIME_S,
    },
    'hardware': {
        'device': DEVICE,
        'gpu_name': gpu_name,
        'gpu_capability': gpu_capability,
        'gpu_total_vram_gb': gpu_total_vram_gb,
        'vram_peak_gb_during_run': VRAM_PEAK_GB,
        'cuda_version': cuda_version,
        'cudnn_version': cudnn_version,
    },
    'software': {
        'python': platform.python_version(),
        'platform': platform.platform(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'pillow': PIL.__version__,
        'numpy': np.__version__,
        'pandas': pd.__version__,
    },
    'seed': SEED,
    'decoding': {
        'do_sample': False,
        'num_beams': 1,
        'enable_thinking': False,
        'max_new_tokens': MAX_NEW_TOKENS,
    },
    'system_prompt': SYSTEM_PROMPT,
    'prompt_variants_evaluated': PROMPT_VARIANTS,
    'datasets': {
        'canonical': {
            'n': int(len(canonical_df)),
            'per_type': {k: int(v) for k, v in canonical_df.groupby('type').size().to_dict().items()},
        },
        'robustness': {
            'n': int(len(robust_df)),
            'per_type': {k: int(v) for k, v in robust_df.groupby('type').size().to_dict().items()},
        },
    },
    'total_inference_time_s': TOTAL_INFERENCE_TIME_S,
    'primary_metrics_per_variant': summary_df.to_dict(orient='records'),
    'primary_metrics_per_variant_per_type': type_df.to_dict(orient='records'),
    'text_similarity_metrics_per_variant': support_df.to_dict(orient='records'),
}

with (RESULTS_DIR / 'reproducibility.json').open('w', encoding='utf-8') as f:
    json.dump(reproducibility, f, indent=2)

print('Reproducibility log written. Snapshot:')
print(f'  Model:    {MODEL_ID} ({N_PARAMS_BILLIONS}B, {MODEL_DTYPE})')
print(f'  GPU:      {gpu_name} ({gpu_total_vram_gb} GB, peak used: {VRAM_PEAK_GB} GB)')
print(f'  Seed:     {SEED}')
print(f'  Software: torch {torch.__version__} | transformers {transformers.__version__} | python {platform.python_version()}')
print(f'  Run UTC:  {reproducibility["run_timestamp_utc"]}')

## 9. Comparison table and results bundle

Prints the head-to-head comparison, lists the files written to
`/content/results/`, zips them and downloads the bundle.

In [ ]:
import zipfile

print('=' * 76)
print(f'SYMBOLIC GROUNDING (TEXT TO PDDL) - {MODEL_ID}')
print('=' * 76)
print('\nPrimary metrics:')
print(summary_df.to_string(index=False))
print('\nPer-type breakdown:')
print(type_df.to_string(index=False))
print('\nText similarity metrics:')
print(support_df.to_string(index=False))
print()
print(f'Model load:        {MODEL_LOAD_TIME_S} s')
print(f'Peak VRAM:         {VRAM_PEAK_GB} GB ({gpu_name})')
print(f'Total inference:   {TOTAL_INFERENCE_TIME_S} s ({len(scored_df)} inferences)')
print()
print(f'Artefacts written to {RESULTS_DIR}:')
for p in sorted(RESULTS_DIR.iterdir()):
    print(f'  - {p.name}')
print('=' * 76)

bundle_path = BASE_DIR / 'text_to_pddl_validation_results.zip'
with zipfile.ZipFile(bundle_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RESULTS_DIR.rglob('*')):
        if p.is_file():
            z.write(p, arcname=f'results/{p.relative_to(RESULTS_DIR).as_posix()}')

print(f'\nBundled results -> {bundle_path} ({bundle_path.stat().st_size / 1024:.1f} KB)')

try:
    from google.colab import files  # type: ignore
    files.download(str(bundle_path))
    print('Browser download triggered.')
except Exception as e:
    print(f'(Not in Colab or download failed: {e}). Find the bundle at {bundle_path}.')